In [ ]:
# CELL 1: Install dependencies!pip install -q unsloth trl peft datasets accelerate bitsandbytes huggingface_hub

In [ ]:
# CELL 2: Download model (NO TOKEN NEEDED - Apache 2.0)import osos.system("git lfs install")os.system("git clone https://huggingface.co/Qwen/Qwen3-4B-Thinking-2507 /content/model")

In [ ]:
# CELL 3: Mount Google Drive and load datafrom google.colab import drivedrive.mount('/content/drive')import shutilshutil.copy('/content/drive/MyDrive/grpo_train_ready.jsonl', '/content/data.jsonl')

In [ ]:
# CELL 4: Load and prepare datasetfrom datasets import Datasetimport jsondata = []with open('/content/data.jsonl') as f:    for line in f:        data.append(json.loads(line.strip()))dataset = Dataset.from_list(data)print(f"Loaded {len(dataset)} training examples")

In [ ]:
# CELL 5: Load model with Unsloth (4-bit for T4)from unsloth import FastLanguageModelimport torchmodel, tokenizer = FastLanguageModel.from_pretrained(    model_name="/content/model",    max_seq_length=1024,    load_in_4bit=True,    dtype=torch.bfloat16,)model = FastLanguageModel.get_peft_model(    model,    r=16,    lora_alpha=32,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],    lora_dropout=0.05,    bias="none",    use_gradient_checkpointing="unsloth",)

In [ ]:
# CELL 6: Define reward functionsdef format_reward(completions, **kwargs):    return [1.0 if "<reasoning>" in c else 0.0 for c in completions]def solution_reward(completions, **kwargs):    return [1.0 if "<solution>" in c else 0.0 for c in completions]

In [ ]:
# CELL 7: Configure GRPO trainingfrom trl import GRPOConfig, GRPOTrainertraining_args = GRPOConfig(    output_dir="/content/output",    num_train_epochs=3,    per_device_train_batch_size=4,    gradient_accumulation_steps=4,    learning_rate=1e-5,    warmup_ratio=0.03,    lr_scheduler_type="cosine",    logging_steps=10,    save_steps=50,    save_total_limit=3,    bf16=True,    beta=0.01,    max_prompt_length=512,    max_completion_length=512,    report_to="none",)

In [ ]:
# CELL 8: Train!trainer = GRPOTrainer(    model=model,    processing_class=tokenizer,    reward_funcs=[format_reward, solution_reward],    args=training_args,    train_dataset=dataset,)trainer.train()print("Training complete!")

In [ ]:
# CELL 9: Save trained model to Drivemodel.save_pretrained("/content/drive/MyDrive/bionic-daughter-grpo")tokenizer.save_pretrained("/content/drive/MyDrive/bionic-daughter-grpo")print("Model saved to Google Drive!")